# SentinelVision AI
## Notebook 05B — Supervised Classifier on VideoMAE Embeddings

### Objective

Notebook 03 created VideoMAE embeddings from real surveillance video clips.

Notebook 04 tested unsupervised anomaly detection models on those embeddings.

This notebook tests whether the VideoMAE embeddings can support supervised binary classification:

`normal clip` vs `anomalous-source clip`

The model is not fine-tuning VideoMAE itself.

Instead, VideoMAE stays frozen and we train classifiers on top of the extracted embeddings.

In [1]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

Load Embeddings

In [2]:
embeddings_path = Path("../data/embeddings/clip_embeddings_videomae_sample.parquet")

embeddings = pd.read_parquet(embeddings_path)

print(embeddings.shape)
embeddings.head()

(600, 775)


,clip_id,video_id,category,binary_label,split,start_seconds,end_seconds,feature_000,feature_001,feature_002,...,feature_758,feature_759,feature_760,feature_761,feature_762,feature_763,feature_764,feature_765,feature_766,feature_767
0,Normal_Videos_365_x264_clip_00003,Normal_Videos_365_x264,Normal,0,test,12.0,16.0,0.178769,3.729066,3.666701,...,2.413040,-1.240642,-1.159237,-1.608377,3.872948,1.565241,0.879731,0.846557,-1.191670,-0.930276
1,Normal_Videos_365_x264_clip_00030,Normal_Videos_365_x264,Normal,0,test,120.0,124.0,0.278189,2.779260,3.542873,...,-0.741295,-1.844295,0.887464,0.550726,3.069808,1.138670,1.094854,0.417987,-1.605511,1.888869
2,Normal_Videos_365_x264_clip_00032,Normal_Videos_365_x264,Normal,0,test,128.0,132.0,0.229807,3.365491,3.970525,...,1.338005,-2.018188,-0.082708,-0.061900,3.764704,1.424313,0.843749,1.016206,-1.329999,1.080641
3,Normal_Videos_641_x264_clip_00015,Normal_Videos_641_x264,Normal,0,test,60.0,64.0,-5.002384,1.898196,0.756616,...,-2.385197,0.084327,4.650817,3.130729,2.914249,-0.534147,2.499364,3.860784,0.763254,-1.158123
4,Normal_Videos_312_x264_clip_00000,Normal_Videos_312_x264,Normal,0,test,0.0,4.0,-4.661326,1.897303,1.866576,...,0.865088,-1.091096,1.328566,4.970067,2.660053,1.487340,2.988191,1.090254,0.692304,0.847159


#### Verify balance

In [3]:
embeddings.groupby(["split", "binary_label"]).size()

split       binary_label
test        0               100
            1               100
train       0               100
            1               100
validation  0               100
            1               100
dtype: int64

In [4]:
feature_columns = [
    column
    for column in embeddings.columns
    if column.startswith("feature_")
]

len(feature_columns)

768

In [5]:
train_data = embeddings[
    embeddings["split"] == "train"
].copy()

validation_data = embeddings[
    embeddings["split"] == "validation"
].copy()

test_data = embeddings[
    embeddings["split"] == "test"
].copy()

print("Train:", train_data.shape)
print("Validation:", validation_data.shape)
print("Test:", test_data.shape)

Train: (200, 775)
Validation: (200, 775)
Test: (200, 775)


In [6]:
features_train = train_data[feature_columns]
target_train = train_data["binary_label"]

features_valid = validation_data[feature_columns]
target_valid = validation_data["binary_label"]

features_test = test_data[feature_columns]
target_test = test_data["binary_label"]

print(features_train.shape)
print(features_valid.shape)
print(features_test.shape)

(200, 768)
(200, 768)
(200, 768)


Helper function for evaluation

In [7]:
def evaluate_classifier(
    model_name,
    model,
    features,
    target,
):
    """
    Evaluate a supervised classifier using predicted probabilities.

    The positive class is 1, meaning anomalous-source clip.
    """
    probabilities = model.predict_proba(features)[:, 1]

    precision, recall, thresholds = precision_recall_curve(
        target,
        probabilities,
    )

    f1_scores = []

    for threshold in thresholds:
        predictions = (
            probabilities >= threshold
        ).astype(int)

        score = f1_score(
            target,
            predictions,
            zero_division=0,
        )

        f1_scores.append(score)

    best_index = int(
        max(
            range(len(f1_scores)),
            key=f1_scores.__getitem__,
        )
    )

    best_threshold = thresholds[best_index]

    predictions = (
        probabilities >= best_threshold
    ).astype(int)

    results = {
        "model": model_name,
        "roc_auc": roc_auc_score(
            target,
            probabilities,
        ),
        "pr_auc": average_precision_score(
            target,
            probabilities,
        ),
        "f1": f1_score(
            target,
            predictions,
            zero_division=0,
        ),
        "threshold": best_threshold,
    }

    print(model_name)
    print(results)
    print()
    print(
        classification_report(
            target,
            predictions,
            zero_division=0,
        )
    )
    print("Confusion matrix:")
    print(
        confusion_matrix(
            target,
            predictions,
        )
    )

    return results

#### Train Logistic Regression

In [8]:
logistic_regression_model = Pipeline(
    steps=[
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

logistic_regression_model.fit(
    features_train,
    target_train,
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](768,)","['feature_000','feature_001','feature_002',...,'feature_765','feature_766', 'feature_767']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,768
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [9]:
logistic_regression_valid_results = evaluate_classifier(
    model_name="Logistic Regression on VideoMAE Embeddings",
    model=logistic_regression_model,
    features=features_valid,
    target=target_valid,
)

logistic_regression_valid_results

Logistic Regression on VideoMAE Embeddings
{'model': 'Logistic Regression on VideoMAE Embeddings', 'roc_auc': 0.9469000000000001, 'pr_auc': 0.952224199361345, 'f1': 0.898989898989899, 'threshold': np.float64(0.9389742904287082)}

              precision    recall  f1-score   support

           0       0.89      0.91      0.90       100
           1       0.91      0.89      0.90       100

    accuracy                           0.90       200
   macro avg       0.90      0.90      0.90       200
weighted avg       0.90      0.90      0.90       200

Confusion matrix:
[[91  9]
 [11 89]]


{'model': 'Logistic Regression on VideoMAE Embeddings',
 'roc_auc': 0.9469000000000001,
 'pr_auc': 0.952224199361345,
 'f1': 0.898989898989899,
 'threshold': np.float64(0.9389742904287082)}

### Train Random Forest

In [16]:
random_forest_model = RandomForestClassifier(
    class_weight="balanced",
    random_state=42
)

random_forest_model.fit(
    features_train,
    target_train,
)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.

In [17]:
random_forest_valid_results = evaluate_classifier(
    model_name="Random Forest on VideoMAE Embeddings",
    model=random_forest_model,
    features=features_valid,
    target=target_valid,
)

random_forest_valid_results

Random Forest on VideoMAE Embeddings
{'model': 'Random Forest on VideoMAE Embeddings', 'roc_auc': 0.9014, 'pr_auc': 0.9204095675648034, 'f1': 0.8374384236453202, 'threshold': np.float64(0.62)}

              precision    recall  f1-score   support

           0       0.85      0.82      0.83       100
           1       0.83      0.85      0.84       100

    accuracy                           0.83       200
   macro avg       0.84      0.83      0.83       200
weighted avg       0.84      0.83      0.83       200

Confusion matrix:
[[82 18]
 [15 85]]


{'model': 'Random Forest on VideoMAE Embeddings',
 'roc_auc': 0.9014,
 'pr_auc': 0.9204095675648034,
 'f1': 0.8374384236453202,
 'threshold': np.float64(0.62)}

#### Train HistGradientBoostingClassifier

In [22]:
hist_gradient_boosting_model = HistGradientBoostingClassifier(
    random_state=42,
)

hist_gradient_boosting_model.fit(
    features_train,
    target_train,
)

,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.1
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",100
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""Categorical"" and ""Enum"" are considered to be categorical features. The input must be a dataframe that is supported by narwhals (or supports it): :func:`narwhals.from_native` must work. This is the case, for instance, for pandas and polars DataFrames.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing val

In [23]:
hist_gradient_boosting_valid_results = evaluate_classifier(
    model_name="HistGradientBoosting on VideoMAE Embeddings",
    model=hist_gradient_boosting_model,
    features=features_valid,
    target=target_valid,
)

hist_gradient_boosting_valid_results

HistGradientBoosting on VideoMAE Embeddings
{'model': 'HistGradientBoosting on VideoMAE Embeddings', 'roc_auc': 0.9064, 'pr_auc': 0.9262648842080112, 'f1': 0.8350515463917526, 'threshold': np.float64(0.9010280722596212)}

              precision    recall  f1-score   support

           0       0.82      0.87      0.84       100
           1       0.86      0.81      0.84       100

    accuracy                           0.84       200
   macro avg       0.84      0.84      0.84       200
weighted avg       0.84      0.84      0.84       200

Confusion matrix:
[[87 13]
 [19 81]]


{'model': 'HistGradientBoosting on VideoMAE Embeddings',
 'roc_auc': 0.9064,
 'pr_auc': 0.9262648842080112,
 'f1': 0.8350515463917526,
 'threshold': np.float64(0.9010280722596212)}

#### Compare validation results

In [24]:
validation_results = pd.DataFrame(
    [
        logistic_regression_valid_results,
        random_forest_valid_results,
        hist_gradient_boosting_valid_results,
    ]
)

validation_results.sort_values(
    by="pr_auc",
    ascending=False,
)

,model,roc_auc,pr_auc,f1,threshold
0,Logistic Regression on VideoMAE Embeddings,0.9469,0.952224,0.898990,0.938974
2,HistGradientBoosting on VideoMAE Embeddings,0.9064,0.926265,0.835052,0.901028
1,Random Forest on VideoMAE Embeddings,0.9014,0.920410,0.837438,0.620000


#### best validation model

In [25]:
best_validation_result = validation_results.sort_values(
    by="pr_auc",
    ascending=False,
).iloc[0]

best_validation_result

model        Logistic Regression on VideoMAE Embeddings
roc_auc                                          0.9469
pr_auc                                         0.952224
f1                                              0.89899
threshold                                      0.938974
Name: 0, dtype: object

In [26]:
trained_models = {
    "Logistic Regression on VideoMAE Embeddings": logistic_regression_model,
    "Random Forest on VideoMAE Embeddings": random_forest_model,
    "HistGradientBoosting on VideoMAE Embeddings": hist_gradient_boosting_model,
}

best_model_name = best_validation_result["model"]

best_model = trained_models[
    best_model_name
]

best_model_name

'Logistic Regression on VideoMAE Embeddings'

#### Use the validation threshold from the best model

In [27]:
best_threshold = best_validation_result["threshold"]

best_threshold

np.float64(0.9389742904287082)

Evaluate the best model on the test set

In [28]:
test_probabilities = best_model.predict_proba(
    features_test
)[:, 1]

test_predictions = (
    test_probabilities >= best_threshold
).astype(int)

In [29]:
supervised_test_results = {
    "model": best_model_name,
    "roc_auc": roc_auc_score(
        target_test,
        test_probabilities,
    ),
    "pr_auc": average_precision_score(
        target_test,
        test_probabilities,
    ),
    "f1": f1_score(
        target_test,
        test_predictions,
        zero_division=0,
    ),
    "threshold": best_threshold,
}

supervised_test_results

{'model': 'Logistic Regression on VideoMAE Embeddings',
 'roc_auc': 0.9020999999999999,
 'pr_auc': 0.877501454769589,
 'f1': 0.8282828282828283,
 'threshold': np.float64(0.9389742904287082)}

#### Classification report

In [30]:
print(
    classification_report(
        target_test,
        test_predictions,
        zero_division=0,
    )
)

              precision    recall  f1-score   support

           0       0.82      0.84      0.83       100
           1       0.84      0.82      0.83       100

    accuracy                           0.83       200
   macro avg       0.83      0.83      0.83       200
weighted avg       0.83      0.83      0.83       200



In [31]:
confusion_matrix(
    target_test,
    test_predictions,
)

array([[84, 16],
       [18, 82]])

In [32]:
metrics_dir = Path("../artifacts/metrics")
models_dir = Path("../models/production")

metrics_dir.mkdir(
    parents=True,
    exist_ok=True,
)

models_dir.mkdir(
    parents=True,
    exist_ok=True,
)

In [33]:
supervised_results = pd.DataFrame(
    [supervised_test_results]
)

supervised_results.to_csv(
    metrics_dir / "supervised_videomae_results.csv",
    index=False,
)

joblib.dump(
    best_model,
    models_dir / "supervised_videomae_classifier.joblib",
)

supervised_results

,model,roc_auc,pr_auc,f1,threshold
0,Logistic Regression on VideoMAE Embeddings,0.9021,0.877501,0.828283,0.938974


#### Compare against Notebook 04

In [34]:
baseline_results = pd.read_csv(
    metrics_dir / "baseline_model_results.csv"
)

combined_results = pd.concat(
    [
        baseline_results,
        supervised_results,
    ],
    ignore_index=True,
)

combined_results.sort_values(
    by="pr_auc",
    ascending=False,
)

,model,roc_auc,pr_auc,f1,threshold
2,Logistic Regression on VideoMAE Embeddings,0.9021,0.877501,0.828283,0.938974
0,Distance Threshold Baseline,0.6950,0.774326,0.628975,NaN
1,Isolation Forest,0.6659,0.732748,0.636364,NaN
